# 00a -- DueCare Prompt Prioritizer (Data Pipeline)

**DueCare** | Named for Cal. Civ. Code sect. 1714(a)

---

**Purpose:** Select a diverse, high-impact subset from 74,567 trafficking
prompts for evaluation against Gemma 4.

| | |
|---|---|
| **Input** | `seed_prompts.jsonl` (74,567 prompts, 63.5 MB) from the DueCare trafficking domain pack |
| **Output** | `curated_prompts.jsonl` (~2,000 prompts, balanced across categories and difficulty) |
| **Prerequisites** | `duecare-llm-wheels` dataset attached; no GPU required |
| **Pipeline position** | Stage 1 of the DueCare data pipeline. Previous: none. Next: NB 00b (Prompt Remixer). |

---

### Why prioritization is needed

The full corpus is too large to run through Gemma in one session
(74K prompts x ~5s each = 103 hours on T4 GPU). We need a curated
subset that maximizes the information value of every prompt we evaluate.

### Prioritization strategy

1. **Graded first** -- 204 prompts with 5-level reference responses (highest value for calibration)
2. **Category coverage** -- ensure all 5 rubric categories are represented
3. **Difficulty balance** -- basic / medium / hard in roughly equal proportions
4. **Source diversity** -- manual > legacy > generated > untested
5. **Length filtering** -- drop prompts < 20 chars or > 10,000 chars
6. **Near-duplicate removal** -- skip prompts whose first 100 chars match an existing one

### Flow diagram

```
seed_prompts.jsonl (74,567)
         |
         v
  Length filter (20-10K chars)
         |
         v
  Tier 1: All 204 graded prompts
         |
         v
  Tier 2: Category-balanced fill
  (min 100 per primary category)
         |
         v
  Near-duplicate removal
         |
         v
  curated_prompts.jsonl (~2,000)
       feeds NB 00b + NB 00
```


In [ ]:
# Plotly renderer configuration for Kaggle inline display
try:
    import plotly.io as pio
    # 'notebook_connected' pulls plotly.js from CDN — works in Kaggle UI
    pio.renderers.default = 'notebook_connected'
except Exception:
    pass  # plotly not installed — charts in this notebook will use matplotlib


## 1. Install DueCare + load the full corpus

We install DueCare from pre-built wheels, then load the full
74,567-prompt corpus from the trafficking domain pack.


In [ ]:
import subprocess, glob
for p in ['/kaggle/input/duecare-llm-wheels/*.whl',
          '/kaggle/input/datasets/taylorsamarel/duecare-llm-wheels/*.whl',
          '/kaggle/input/**/*.whl']:
    wheels = glob.glob(p, recursive=True)
    if wheels: break
if wheels:
    subprocess.check_call(['pip', 'install'] + wheels + ['--quiet'])


In [ ]:
import json
from pathlib import Path
from collections import Counter, defaultdict

# Load the full corpus
from duecare.domains import register_discovered, load_domain_pack
register_discovered()
pack = load_domain_pack('trafficking')

all_prompts = list(pack.seed_prompts())
print(f'Total prompts in corpus: {len(all_prompts):,}')

# Quick stats
cats = Counter(p.get('category', 'unknown') for p in all_prompts)
graded = [p for p in all_prompts if p.get('graded_responses')]
print(f'Graded (with reference responses): {len(graded)}')
print(f'Unique categories: {len(cats)}')
print(f'\nTop 15 categories:')
for cat, n in cats.most_common(15):
    print(f'  {cat:<45} {n:>7}')


## 2. Define the 5 primary rubric categories

The 85 subcategories in the corpus map to 5 primary rubric categories.
These categories represent the main vectors through which LLMs can
fail on trafficking-related content. Ensuring coverage across all 5
is essential for a meaningful baseline evaluation.

| Category | What it tests |
|---|---|
| **business_framed_exploitation** | Exploitation disguised as business practice |
| **jurisdictional_hierarchy_exploitation** | Cross-border regulatory gaps |
| **financial_crime_blindness** | Fee laundering, debt structures |
| **prompt_injection_amplification** | Adversarial jailbreak attacks |
| **victim_revictimization** | Re-traumatization, coercion normalization |


In [ ]:
# Map the 85 subcategories to the 5 primary rubric categories
PRIMARY_CATEGORIES = {
    'business_framed_exploitation': [
        'business_framed_exploitation', 'labor_trafficking',
        'recruitment_fees', 'debt_bondage', 'wage_theft',
        'contract_fraud', 'subcontracting_evasion',
        'franchise exploitation networks', 'complex_corporate_hierarchy',
        'integrated service complexes', 'insurance and housing schemes',
    ],
    'jurisdictional_hierarchy_exploitation': [
        'jurisdictional_hierarchy_exploitation', 'regulatory_evasion',
        'corridor_specific', 'document_control', 'technology_control',
        'regulatory_evasion_renaming', 'regulatory_evasion_jurisdictional',
        'regulatory_evasion_third_country', 'regulatory_evasion_micro_fees',
        'regulatory_evasion_debt_assignment', 'regulatory_complexity',
    ],
    'financial_crime_blindness': [
        'financial_crime_blindness', 'financial_obfuscation',
        'multi-entity fee laundering', 'circular debt structures',
        'financial_obfuscation_investment', 'financial_obfuscation_fee_transfer',
        'financial_obfuscation_cost_sharing', 'financial_obfuscation_combo',
        'wage_exploitation',
    ],
    'prompt_injection_amplification': [
        'prompt_injection_amplification', 'adversarial',
        'keyword_mutation', 'output_conditioning', 'obfuscation',
        'persona_framed', 'escalation', 'variation',
        'creative_attack', 'cognitive_manipulation',
        'token-level obfuscation', 'encoding obfuscation',
        'context window exploitation', 'semantic drift attack',
        'multi_turn_attack', 'escalation_strategy',
        'hypothetical_scenario', 'multi_step_conversation',
        'psychological_manipulation', 'authority chain attack',
        'socratic method exploitation', 'translation confusion attack',
        'technical jargon obfuscation',
    ],
    'victim_revictimization': [
        'victim_revictimization', 'bystander_dilemma',
        'coercion', 'coercion_debt_trap_savior', 'coercion_family_pressure',
        'coercion_document_confiscation', 'coercion_graduated', 'coercion_false_hope',
        'justification_coercion', 'justification_emergency',
        'justification_normalization', 'justification_ideological',
        'justification_religious', 'justification_rationalization',
        'justification_destitution', 'justification_authority',
        'moral_religious_framing', 'moral_religious_biblical',
        'moral_religious_cultural', 'moral_religious_philosophical',
        'moral_religious_duty_honor', 'moral_religious_virtue',
        'exploiter_framed', 'synthetic victim testimony',
    ],
}

# Build reverse lookup
SUBCAT_TO_PRIMARY = {}
for primary, subcats in PRIMARY_CATEGORIES.items():
    for sub in subcats:
        SUBCAT_TO_PRIMARY[sub] = primary

def get_primary_category(prompt):
    cat = prompt.get('category', 'unknown')
    return SUBCAT_TO_PRIMARY.get(cat, 'other')

# Classify all prompts
primary_dist = Counter(get_primary_category(p) for p in all_prompts)
print('Primary category distribution:')
for cat, n in primary_dist.most_common():
    print(f'  {cat:<45} {n:>7}')


## 3. Prioritize and select

The selection algorithm works in three passes:
1. **All graded prompts** -- these have reference responses for calibration
2. **Category fill** -- ensure minimum representation per primary category
3. **Remaining slots** -- fill from highest-quality sources

Near-duplicates are removed by comparing the first 100 characters
of each prompt. This prevents wasting evaluation budget on prompts
that differ only in minor phrasing.


In [ ]:
import hashlib

TARGET_SIZE = 2000  # Total curated prompts
MIN_PER_PRIMARY = 100  # Minimum per primary category

# Priority tiers
TIER_1 = []  # Graded (with reference responses)
TIER_2 = defaultdict(list)  # By primary category, source quality

# Source quality ordering (higher = better)
SOURCE_PRIORITY = {
    'taylor_amarel_tests': 10,
    'taylor_amarel_extended': 10,
    'legacy_': 8,
    'all_conversations': 6,
    'gen_': 5,
    'advanced_': 5,
    'test_catalog': 4,
    'trafficking_tests': 4,
    'untested_prompts': 2,
    'claude_cli': 3,
}

def source_score(prompt):
    src = prompt.get('source', '')
    for prefix, score in SOURCE_PRIORITY.items():
        if src.startswith(prefix):
            return score
    return 1

# Length filter
valid = [p for p in all_prompts if 20 <= len(p.get('text', '')) <= 10000]
print(f'After length filter: {len(valid):,} (dropped {len(all_prompts) - len(valid):,})')

# Separate graded vs ungraded
for p in valid:
    if p.get('graded_responses'):
        TIER_1.append(p)
    else:
        primary = get_primary_category(p)
        TIER_2[primary].append(p)

# Sort each tier by source quality
for cat in TIER_2:
    TIER_2[cat].sort(key=source_score, reverse=True)

print(f'\nTier 1 (graded): {len(TIER_1)}')
print(f'Tier 2 by category:')
for cat, items in sorted(TIER_2.items(), key=lambda x: -len(x[1])):
    print(f'  {cat:<45} {len(items):>7}')


In [ ]:
# Build the curated set
curated = []
seen_prefixes = set()

def add_prompt(p):
    """Add prompt if not a near-duplicate."""
    prefix = p['text'][:100].lower().strip()
    if prefix in seen_prefixes:
        return False
    seen_prefixes.add(prefix)
    curated.append(p)
    return True

# Step 1: All graded prompts (highest value)
for p in TIER_1:
    add_prompt(p)
print(f'After graded: {len(curated)}')

# Step 2: Ensure minimum per primary category
remaining = TARGET_SIZE - len(curated)
per_cat = max(MIN_PER_PRIMARY, remaining // max(len(TIER_2), 1))

for cat, items in TIER_2.items():
    added = 0
    for p in items:
        if added >= per_cat:
            break
        if add_prompt(p):
            added += 1

print(f'After category fill: {len(curated)}')

# Step 3: Fill remaining slots from largest categories
if len(curated) < TARGET_SIZE:
    all_remaining = []
    for cat, items in TIER_2.items():
        for p in items:
            if p['text'][:100].lower().strip() not in seen_prefixes:
                all_remaining.append(p)
    all_remaining.sort(key=source_score, reverse=True)
    for p in all_remaining:
        if len(curated) >= TARGET_SIZE:
            break
        add_prompt(p)

print(f'Final curated set: {len(curated)}')


## 4. Curated set statistics

The stats below verify that the curated set meets our targets:
- All graded prompts included (for calibration)
- All 5 primary categories represented (for coverage)
- Difficulty levels balanced (for comprehensive evaluation)
- Total size manageable within a Kaggle GPU session


In [ ]:
curated_cats = Counter(get_primary_category(p) for p in curated)
curated_diff = Counter(p.get('difficulty', 'unknown') for p in curated)
curated_graded = sum(1 for p in curated if p.get('graded_responses'))

print(f'Curated prompts:     {len(curated):,}')
print(f'Graded (with refs):  {curated_graded}')
print(f'Ungraded:            {len(curated) - curated_graded}')
print(f'\nBy primary category:')
for cat, n in curated_cats.most_common():
    pct = n / len(curated) * 100
    print(f'  {cat:<45} {n:>5} ({pct:>5.1f}%)')
print(f'\nBy difficulty:')
for diff, n in curated_diff.most_common():
    print(f'  {diff:<20} {n:>5}')

# Show a few example prompts per category
print(f'\n--- Sample prompts ---')
shown = set()
for p in curated:
    cat = get_primary_category(p)
    if cat in shown:
        continue
    shown.add(cat)
    print(f'\n[{cat}] {p["id"]}')
    print(f'  {p["text"][:150]}...')


In [ ]:
!pip install plotly --quiet

## Corpus analysis -- interactive charts

The following visualizations break down the full 74,567-prompt corpus
by category, grading status, and difficulty. These charts make it
easier to spot imbalances that the selection algorithm must correct.

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Build a dataframe-like structure for the treemap.
# Each prompt has a primary category and a subcategory.
treemap_rows = []
for p in all_prompts:
    subcat = p.get('category', 'unknown')
    primary = get_primary_category(p)
    treemap_rows.append({'primary': primary, 'subcategory': subcat})

# Aggregate counts per (primary, subcategory) pair
from collections import Counter as _Counter
pair_counts = _Counter((r['primary'], r['subcategory']) for r in treemap_rows)

labels = []
parents = []
values = []

# Add primary categories as top-level segments
primary_totals = _Counter(r['primary'] for r in treemap_rows)
for prim, total in primary_totals.most_common():
    labels.append(prim)
    parents.append('')
    values.append(total)

# Add subcategories nested under their primary
for (prim, sub), count in pair_counts.most_common():
    if sub == prim:
        continue  # skip if subcategory name matches primary (already counted)
    label = f'{sub}'
    labels.append(label)
    parents.append(prim)
    values.append(count)

fig_tree = go.Figure(go.Treemap(
    labels=labels,
    parents=parents,
    values=values,
    branchvalues='total',
    textinfo='label+value+percent root',
    hovertemplate='<b>%{label}</b><br>Count: %{value:,}<br>Share of corpus: %{percentRoot:.1%}<extra></extra>',
    marker=dict(
        colors=values,
        colorscale='Tealgrn',
        showscale=False,
    ),
))

fig_tree.update_layout(
    title=dict(text='Full corpus category breakdown (74,567 prompts)', x=0.5),
    margin=dict(t=50, l=10, r=10, b=10),
    height=550,
)
fig_tree.show()

In [ ]:
# Stacked bar chart: graded vs ungraded prompts per primary category.
# Graded prompts have 5-level reference responses and are the most
# valuable for calibrating model evaluation.

bar_data = {}
for p in all_prompts:
    primary = get_primary_category(p)
    is_graded = bool(p.get('graded_responses'))
    if primary not in bar_data:
        bar_data[primary] = {'graded': 0, 'ungraded': 0}
    if is_graded:
        bar_data[primary]['graded'] += 1
    else:
        bar_data[primary]['ungraded'] += 1

# Sort categories by total count descending
sorted_cats = sorted(bar_data.keys(), key=lambda c: bar_data[c]['graded'] + bar_data[c]['ungraded'], reverse=True)
graded_vals = [bar_data[c]['graded'] for c in sorted_cats]
ungraded_vals = [bar_data[c]['ungraded'] for c in sorted_cats]

fig_bar = go.Figure()
fig_bar.add_trace(go.Bar(
    name='Graded (with reference responses)',
    x=sorted_cats,
    y=graded_vals,
    marker_color='#2ca02c',
    hovertemplate='<b>%{x}</b><br>Graded: %{y:,}<extra></extra>',
))
fig_bar.add_trace(go.Bar(
    name='Ungraded',
    x=sorted_cats,
    y=ungraded_vals,
    marker_color='#1f77b4',
    hovertemplate='<b>%{x}</b><br>Ungraded: %{y:,}<extra></extra>',
))

fig_bar.update_layout(
    barmode='stack',
    title=dict(text='Graded vs ungraded prompts per primary category', x=0.5),
    xaxis_title='Primary category',
    yaxis_title='Number of prompts',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
    height=450,
    margin=dict(t=80, b=120),
    xaxis_tickangle=-25,
)
fig_bar.show()

## Selection coverage

The selected 2,000-prompt subset must cover all categories proportionally.
This chart shows how the selection compares to the full corpus. A well-balanced
selection will have similar percentage shares across categories, while
over- or under-representation becomes visible as differences between the
left (full corpus) and right (selected subset) pie charts.

In [ ]:
# Side-by-side pie charts: full corpus vs selected subset category distribution.
# This comparison reveals whether the selection algorithm preserves the
# proportional balance of the original corpus or deliberately rebalances
# underrepresented categories.

corpus_primary = Counter(get_primary_category(p) for p in all_prompts)
curated_primary = Counter(get_primary_category(p) for p in curated)

# Use a consistent ordering and color map so both pies match visually
all_cats_ordered = [c for c, _ in corpus_primary.most_common()]
color_map = {
    'prompt_injection_amplification': '#636EFA',
    'victim_revictimization': '#EF553B',
    'business_framed_exploitation': '#00CC96',
    'jurisdictional_hierarchy_exploitation': '#AB63FA',
    'financial_crime_blindness': '#FFA15A',
    'other': '#B6B6B6',
}
colors_ordered = [color_map.get(c, '#B6B6B6') for c in all_cats_ordered]

fig_pies = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'pie'}, {'type': 'pie'}]],
    subplot_titles=(
        f'Full corpus ({sum(corpus_primary.values()):,} prompts)',
        f'Selected subset ({sum(curated_primary.values()):,} prompts)',
    ),
)

fig_pies.add_trace(
    go.Pie(
        labels=all_cats_ordered,
        values=[corpus_primary[c] for c in all_cats_ordered],
        marker=dict(colors=colors_ordered),
        textinfo='percent+label',
        textposition='inside',
        hovertemplate='<b>%{label}</b><br>Count: %{value:,}<br>Share: %{percent}<extra></extra>',
        showlegend=False,
    ),
    row=1, col=1,
)

fig_pies.add_trace(
    go.Pie(
        labels=all_cats_ordered,
        values=[curated_primary.get(c, 0) for c in all_cats_ordered],
        marker=dict(colors=colors_ordered),
        textinfo='percent+label',
        textposition='inside',
        hovertemplate='<b>%{label}</b><br>Count: %{value:,}<br>Share: %{percent}<extra></extra>',
        showlegend=False,
    ),
    row=1, col=2,
)

fig_pies.update_layout(
    title=dict(text='Category distribution -- full corpus vs selected subset', x=0.5),
    height=450,
    margin=dict(t=80, b=30),
)
fig_pies.show()

## 5. Save curated set

Save the curated subset to `curated_prompts.jsonl` in JSONL format.
This file feeds into the next two stages of the data pipeline.


In [ ]:
import json

output_path = 'curated_prompts.jsonl'
with open(output_path, 'w', encoding='utf-8') as f:
    for p in curated:
        f.write(json.dumps(p, ensure_ascii=False, default=str) + '\n')

print(f'Saved {len(curated):,} curated prompts to {output_path}')
print(f'This file feeds into Notebook 00 (Gemma Exploration).')
print(f'\nTo use in Notebook 00:')
print(f'  prompts = [json.loads(line) for line in open("curated_prompts.jsonl")]')


## Summary and next steps

### What this notebook produces

This notebook produces a balanced, prioritized subset of the full 74,567-prompt
corpus. The curated set includes all 204 graded prompts that carry 5-level
reference responses, which are essential for calibrating the evaluation rubric.
Every one of the 5 primary rubric categories is represented with at least 100
prompts, and difficulty levels are balanced across basic, medium, and hard tiers.
Near-duplicate prompts have been removed to avoid wasting evaluation budget, and
source quality is prioritized so that manually authored prompts rank above
generated or untested ones.

### Connection to the pipeline

The Prompt Remixer (Notebook 00b) takes this curated set as input and generates
adversarial variations using five mutation strategies: academic framing, role-play,
corporate wrapping, urgency pressure, and corridor swaps. Those variations are
then combined with the originals and fed into Notebook 00 (Gemma Exploration),
where stock Gemma 4 E4B scores every response using the weighted rubric.

### Why curation matters

Running the full 74,567-prompt corpus through Gemma would take approximately
103 hours on a T4 GPU. The curated subset captures the same information
diversity in roughly 17 minutes. The prioritization algorithm ensures that every
evaluated prompt contributes maximum information value: graded prompts for
calibration, category coverage for completeness, and difficulty balance for
robustness. This curated foundation is what every subsequent evaluation builds on.